# 种植模块：Gurobi MILP 求解器

本 notebook 实现 `planting/docs/种植模块_最优化问题与约束_审核稿.md` 中的独立种植模型。规划期为 Year 1 的 112 天，目标为最大化第 112 天日末现金。

建模顺序：读取日事件数据 → 创建变量 → 按审核稿编号添加约束 → 求解 → 审计解。

## 0. 环境与许可证

需要 Python 3、`pandas` 和已授权的 `gurobipy`。首次运行可在终端执行 `python -m pip install gurobipy`；随后按 Gurobi 的 academic / WLS / 本地许可证方式配置。若导入失败，下一个代码单元会给出明确提示。

In [ ]:
from pathlib import Path
import pandas as pd

try:
    import gurobipy as gp
    from gurobipy import GRB
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        '未找到 gurobipy。请先运行：python -m pip install gurobipy；并配置有效 Gurobi 许可证。'
    ) from exc

# Notebook 可从项目根目录或 planting/notebooks/ 目录启动。
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
PROJECT_ROOT = next((p for p in candidates if (p / 'planting' / 'data').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('找不到项目根目录：应包含 planting/data/。')

PLANTING_DATA = PROJECT_ROOT / 'planting' / 'data'
SHARED_DATA = PROJECT_ROOT / 'shared' / 'data'
print(f'Project root: {PROJECT_ROOT}')


## 1. 读取输入与构造参数

`crop_daily_events.csv` 是收获、浇水、最终土地释放和每格收入的唯一日事件来源。工具每格系数与升级现金成本均以 `planting_tool_catalog.csv` 为主。

In [ ]:
calendar = pd.read_csv(PLANTING_DATA / 'crop_planting_calendar.csv')
events = pd.read_csv(PLANTING_DATA / 'crop_daily_events.csv')
crops = pd.read_csv(PLANTING_DATA / 'crops.csv')
tools = pd.read_csv(PLANTING_DATA / 'planting_tool_catalog.csv')
global_parameters = pd.read_csv(SHARED_DATA / 'global_parameters.csv')

def global_value(parameter_id, cast=float):
    value = global_parameters.loc[global_parameters.parameter_id.eq(parameter_id), 'value'].iloc[0]
    return cast(value)

T = list(range(1, 113))
C = sorted(calendar.crop_key.unique())
J = ['H', 'W']                 # H = Hoe; W = Watering Can
K = [0, 1, 2, 3]              # Basic, Copper, Steel, Gold
UPGRADE_K = [1, 2, 3]
tier_name = {0: 'Basic', 1: 'Copper', 2: 'Steel', 3: 'Gold'}
tool_name = {'H': 'Hoe', 'W': 'Watering Can'}

# A[c,t] 和种子价格 p[c]。crops.csv 的季节重复行对同一物理种子价格相同。
A = {(r.crop_key, int(r.plant_day)): int(r.plant_allowed_A_ct == 'true')
     for r in calendar.itertuples(index=False)}
seed_price_by_id = crops.groupby('seed_id').seed_price_g.first().to_dict()
seed_id_by_crop = {r.crop_key: int(r.seed_id) for r in calendar.itertuples(index=False)}
p = {c: float(seed_price_by_id[seed_id_by_crop[c]]) for c in C}

# 日事件参数 h, f, w_need, r；没有事件的组合默认取 0。
event_key = ['crop_key', 'plant_day', 'calendar_day']
event_lookup = events.set_index(event_key)
def event_value(c, s, t, column, cast=float):
    key = (c, s, t)
    if key not in event_lookup.index:
        return cast(0)
    return cast(event_lookup.loc[key, column])

h = {(c, s, t): event_value(c, s, t, 'is_harvest', lambda v: int(v == True))
     for c in C for s in T for t in T if s <= t}
f = {(c, s, t): event_value(c, s, t, 'releases_land', lambda v: int(v == True))
     for c in C for s in T for t in T if s <= t}
w_need = {(c, s, t): event_value(c, s, t, 'needs_water', lambda v: int(v == True))
          for c in C for s in T for t in T if s <= t}
revenue = {(c, s, t): event_value(c, s, t, 'base_revenue_g_per_tile_harvest', float)
           for c in C for s in T for t in T if s <= t}

# 工具目录参数：按工具类型和等级取每格时间、体力及升级现金等价成本。
tool_row = tools.set_index(['tool_type', 'tier'])
tau = {(j, k): float(tool_row.loc[(tool_name[j], tier_name[k]), 'time_game_minutes_per_unit'])
       for j in J for k in K}
energy = {(j, k): float(tool_row.loc[(tool_name[j], tier_name[k]), 'energy_per_unit'])
          for j in J for k in K}
kappa = {k: float(tool_row.loc[('Watering Can', tier_name[k]), 'upgrade_cash_equivalent_cost_g'])
         for k in UPGRADE_K}

TIME_LIMIT = global_value('daily_time_limit_game_minutes')
ENERGY_LIMIT = global_value('daily_energy_limit')
INITIAL_CASH = global_value('initial_gold_g')
DELTA = {t: int(t in {1, 29, 57, 85}) for t in T}
M_L = M_W = int(TIME_LIMIT // 1.4)  # 642：安全日度作业格数上限
X_UB = int(TIME_LIMIT // (1.4 + 1.4))  # 321：新种的播种+浇水时间上限
assert M_L == M_W == 642 and X_UB == 321


## 2. 创建变量

变量命名与审核稿一致：`x, L, S_land, H, W, F, U, S_tool, D, Z, till, water, B`。特别地，`S_land` 是空闲耕地，而 `S_tool` 是工具等级状态，避免符号冲突。

In [ ]:
model = gp.Model('stardew_planting_milp')
model.ModelSense = GRB.MAXIMIZE

# 3.1 作物、土地与由日事件决定的派生量。
x = model.addVars(C, T, vtype=GRB.INTEGER, lb=0, ub=X_UB, name='x')
L = model.addVars(T, vtype=GRB.INTEGER, lb=0, name='L')
S_land = model.addVars(T, vtype=GRB.INTEGER, lb=0, name='S_land')
H = model.addVars(C, T, vtype=GRB.INTEGER, lb=0, name='H')
W = model.addVars(T, vtype=GRB.INTEGER, lb=0, name='W')
F = model.addVars(T, vtype=GRB.INTEGER, lb=0, name='F')

# 3.2 工具升级、等级状态、服务期、实际可用性和作业分配。
U = model.addVars(J, UPGRADE_K, T, vtype=GRB.BINARY, name='U')
S_tool = model.addVars(J, K, T, vtype=GRB.BINARY, name='S_tool')
D = model.addVars(J, T, vtype=GRB.BINARY, name='D')
Z = model.addVars(J, K, T, vtype=GRB.BINARY, name='Z')
till = model.addVars(K, T, vtype=GRB.INTEGER, lb=0, name='till')
water = model.addVars(K, T, vtype=GRB.INTEGER, lb=0, name='water')

# 3.3 现金与资源派生量。B[t] 是 t 日末现金，B[0] 是规划开始时现金。
B = model.addVars(range(113), lb=0, name='B')
crop_revenue = model.addVars(T, lb=0, name='CropRevenue')
seed_cost = model.addVars(T, lb=0, name='SeedCost')
upgrade_cost = model.addVars(T, lb=0, name='UpgradeCost')
time_used = model.addVars(T, lb=0, name='TimeUsed')
energy_used = model.addVars(T, lb=0, name='EnergyUsed')


## 3. 约束实现

以下代码块按审核稿第 4–7 节分组。每组均附约束名称，便于使用 Gurobi IIS 或 LP 导出文件定位不可行原因。

In [ ]:
# 4.1 新种植许可：A[c,t]=0 时，x[c,t] 必为 0。
for c in C:
    for t in T:
        model.addConstr(x[c, t] <= X_UB * A[c, t], name=f'plant_allowed[{c},{t}]')

# 4.2 日事件：收获、最终释放、浇水需求和收获收入都由历史批次决定。
for t in T:
    model.addConstrs((
        H[c, t] == gp.quicksum(h[c, s, t] * x[c, s] for s in T if s <= t)
        for c in C), name=f'harvest_event_day_{t}')
    model.addConstr(F[t] == gp.quicksum(f[c, s, t] * x[c, s] for c in C for s in T if s <= t),
                    name=f'land_release[{t}]')
    model.addConstr(W[t] == gp.quicksum(w_need[c, s, t] * x[c, s] for c in C for s in T if s <= t),
                    name=f'water_need[{t}]')
    model.addConstr(crop_revenue[t] == gp.quicksum(revenue[c, s, t] * x[c, s]
                                                  for c in C for s in T if s <= t),
                    name=f'crop_revenue[{t}]')

# 4.3 空闲耕地平衡：季首日清空上一季的空闲地，但不影响在地跨季作物。
for t in T:
    previous_empty_land = 0 if t == 1 else S_land[t - 1]
    model.addConstr(S_land[t] == (1 - DELTA[t]) * previous_empty_land + L[t] + F[t]
                    - gp.quicksum(x[c, t] for c in C), name=f'land_balance[{t}]')

# 5.1–5.3 工具状态、顺序升级和两天服务期。
for j in J:
    model.addConstr(S_tool[j, 0, 1] == 1, name=f'initial_basic[{j}]')
    for k in UPGRADE_K:
        model.addConstr(S_tool[j, k, 1] == 0, name=f'initial_no_upgrade[{j},{k}]')
    for t in T:
        model.addConstr(gp.quicksum(S_tool[j, k, t] for k in K) == 1,
                        name=f'exact_tool_tier[{j},{t}]')
        for k in UPGRADE_K:
            model.addConstr(U[j, k, t] <= S_tool[j, k - 1, t],
                            name=f'sequential_upgrade[{j},{k},{t}]')
        u_today = gp.quicksum(U[j, k, t] for k in UPGRADE_K)
        u_yesterday = 0 if t == 1 else gp.quicksum(U[j, k, t - 1] for k in UPGRADE_K)
        model.addConstr(D[j, t] == u_today + u_yesterday, name=f'service_state[{j},{t}]')
        model.addConstr(u_today + u_yesterday <= 1, name=f'no_resubmit[{j},{t}]')
        if t >= 2:
            for k in K:
                leave_old = 0 if k == 3 or t == 2 else U[j, k + 1, t - 2]
                enter_new = 0 if k == 0 or t == 2 else U[j, k, t - 2]
                model.addConstr(S_tool[j, k, t] == S_tool[j, k, t - 1] - leave_old + enter_new,
                                name=f'tool_state_transition[{j},{k},{t}]')

# 每日最多交付一把工具；服务期重叠在当前简化下允许。
for t in T:
    model.addConstr(gp.quicksum(U[j, k, t] for j in J for k in UPGRADE_K) <= 1,
                    name=f'one_tool_submission_per_day[{t}]')

# 5.4 实际可用性：送修当天和次日的所有 Z 都为 0。
for j in J:
    for k in K:
        for t in T:
            model.addConstr(Z[j, k, t] <= S_tool[j, k, t], name=f'available_only_owned[{j},{k},{t}]')
            model.addConstr(Z[j, k, t] <= 1 - D[j, t], name=f'not_in_service[{j},{k},{t}]')
            model.addConstr(Z[j, k, t] >= S_tool[j, k, t] - D[j, t],
                            name=f'available_when_owned[{j},{k},{t}]')

# 6 工具行动连接：所有整地和浇水需求都须由当天可用等级完成。
for t in T:
    model.addConstr(gp.quicksum(till[k, t] for k in K) == L[t], name=f'till_all_new_land[{t}]')
    model.addConstr(gp.quicksum(water[k, t] for k in K) == W[t], name=f'water_all_required[{t}]')
    for k in K:
        model.addConstr(till[k, t] <= M_L * Z['H', k, t], name=f'till_requires_hoe[{k},{t}]')
        model.addConstr(water[k, t] <= M_W * Z['W', k, t], name=f'water_requires_can[{k},{t}]')

    # 6 时间与体力。播种、收获每格均为 1.4 分钟且不耗体力。
    model.addConstr(time_used[t] == gp.quicksum(tau['H', k] * till[k, t] for k in K)
                    + 1.4 * gp.quicksum(x[c, t] for c in C)
                    + gp.quicksum(tau['W', k] * water[k, t] for k in K)
                    + 1.4 * gp.quicksum(H[c, t] for c in C), name=f'time_accounting[{t}]')
    model.addConstr(energy_used[t] == gp.quicksum(energy['H', k] * till[k, t] for k in K)
                    + gp.quicksum(energy['W', k] * water[k, t] for k in K),
                    name=f'energy_accounting[{t}]')
    model.addConstr(time_used[t] <= TIME_LIMIT, name=f'time_limit[{t}]')
    model.addConstr(energy_used[t] <= ENERGY_LIMIT, name=f'energy_limit[{t}]')

# 7 现金：日初必须先支付种子／升级；收获收入在日末进入 B[t]。
model.addConstr(B[0] == INITIAL_CASH, name='initial_cash')
for t in T:
    model.addConstr(seed_cost[t] == gp.quicksum(p[c] * x[c, t] for c in C), name=f'seed_cost[{t}]')
    model.addConstr(upgrade_cost[t] == gp.quicksum(kappa[k] * U[j, k, t]
                                                for j in J for k in UPGRADE_K),
                    name=f'upgrade_cost[{t}]')
    model.addConstr(B[t - 1] >= seed_cost[t] + upgrade_cost[t], name=f'cash_before_spending[{t}]')
    model.addConstr(B[t] == B[t - 1] - seed_cost[t] - upgrade_cost[t] + crop_revenue[t],
                    name=f'cash_balance[{t}]')

# 8 目标：最大化 Day 112 日末现金。
model.setObjective(B[112], GRB.MAXIMIZE)
model.update()
print(f'Variables: {model.NumVars:,}; constraints: {model.NumConstrs:,}')


## 4. 求解、结果表与可行性审计

求解后输出逐日资源表、种植汇总和关键约束余量。若模型不可行，代码会导出 `planting_infeasible.ilp`，可在 Gurobi 中检查 IIS。

In [ ]:
model.Params.MIPGap = 0.001
model.optimize()

if model.Status == GRB.OPTIMAL or model.Status == GRB.TIME_LIMIT and model.SolCount > 0:
    print(f'Status: {model.Status}; objective (Day 112 cash): {B[112].X:,.2f}g')

    daily_rows = []
    for t in T:
        submissions = [f'{tool_name[j]}->{tier_name[k]}' for j in J for k in UPGRADE_K if U[j, k, t].X > 0.5]
        daily_rows.append({
            'day': t, 'new_plants': sum(x[c, t].X for c in C), 'new_tilled': L[t].X,
            'watered': W[t].X, 'harvested': sum(H[c, t].X for c in C), 'released_land': F[t].X,
            'empty_land_end': S_land[t].X, 'seed_cost_g': seed_cost[t].X,
            'upgrade_cost_g': upgrade_cost[t].X, 'revenue_g': crop_revenue[t].X,
            'time_minutes': time_used[t].X, 'energy': energy_used[t].X,
            'cash_end_g': B[t].X, 'upgrade_submitted': ', '.join(submissions) or '-'
        })
    daily_solution = pd.DataFrame(daily_rows)
    display(daily_solution.head(20))

    planting_rows = [{'crop_key': c, 'plant_day': t, 'tiles': x[c, t].X}
                     for c in C for t in T if x[c, t].X > 0.5]
    planting_plan = pd.DataFrame(planting_rows).sort_values(['plant_day', 'crop_key'])
    display(planting_plan)

    # 审计：检验日度上限、现金不为负、每日送修次数和工具可用性。
    assert daily_solution.time_minutes.max() <= TIME_LIMIT + 1e-6
    assert daily_solution.energy.max() <= ENERGY_LIMIT + 1e-6
    assert daily_solution.cash_end_g.min() >= -1e-6
    assert all(sum(U[j, k, t].X for j in J for k in UPGRADE_K) <= 1 + 1e-6 for t in T)
    print('Audit passed: time, energy, cash non-negativity, and one-tool-per-day constraints hold.')
else:
    print(f'No usable solution. Gurobi status: {model.Status}')
    if model.Status == GRB.INFEASIBLE:
        model.computeIIS()
        model.write('planting_infeasible.ilp')
        print('Wrote IIS to planting_infeasible.ilp for diagnosis.')
